In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from convert import convert_conllu_list_to_csv
from utils import *

In [2]:
manifest = [
    ("ud",      "gold", "../datasets/ud/test.conllu"),
    ("ud",      "pred", "../out/ud/test.pred.conllu"),
    ("ud-new",  "gold", "../datasets/ud-new/test.conllu"),
    ("ud-new",  "pred", "../out/ud-new/test.pred.conllu"),
    ("ud-old",  "gold", "../datasets/ud-old/test.conllu"),
    ("ud-old",  "pred", "../out/ud-old/test.pred.conllu"),
    ("str",     "gold", "../datasets/str/test.conllu"),
    ("str",     "pred", "../out/str/test.pred.conllu"),
    ("str-new", "gold", "../datasets/str-new/test.conllu"),
    ("str-new", "pred", "../out/str-new/test.pred.conllu"),
    ("str-old", "gold", "../datasets/str-old/test.conllu"),
    ("str-old", "pred", "../out/str-old/test.pred.conllu"),
]

inputs = [p for _, _, p in manifest]
csvs = convert_conllu_list_to_csv(inputs, "csvs")

csv_index = {}
for (split, role, _), csv in zip(manifest, csvs):
    csv_index.setdefault(split, {})[role] = Path(csv)

DATA = {split: load_pair(paths["gold"], paths["pred"]) for split, paths in csv_index.items()}
DATA["ud-new"].head(1)


,,text,form,deprel,upos_g,upos_p,feats_g,feats_p
sent_id,id,,,,,,,
100_2016Tselina_glazami_pervokursnika,1,"Штурвальный, следуя неровностям поля, должен т...",Штурвальный,nsubj,NOUN,ADJ,Animacy=Anim|Case=Nom|Gender=Masc|Number=Sing,Case=Nom|Degree=Pos|Gender=Masc|Number=Sing


In [3]:
ERR = {}
for split, df in DATA.items():
    d = add_flags(df)
    ERR[split] = d.loc[~d["eq_all"]].copy()

summary = pd.concat({k: split_summary(v) for k, v in DATA.items()}, axis=1).T.sort_values("AllTags_acc", ascending=False)
summary

,rows,UPOS_acc,FEATS_acc,AllTags_acc
str,157706.0,0.9935,0.9815,0.9790
str-old,117320.0,0.9935,0.9803,0.9779
ud-old,117320.0,0.9914,0.9750,0.9729
str-new,40386.0,0.9913,0.9754,0.9719
ud-new,40386.0,0.9893,0.9687,0.9660
ud,157706.0,0.9890,0.9652,0.9626


# UPOS

In [4]:
import pandas as pd
def display_side_by_side(freq_dict, n=10):
    frames = []
    for name, df in freq_dict.items():
        tmp = df.head(n).reset_index(drop=True)
        tmp = tmp.add_prefix(name + "_")
        frames.append(tmp)
    combined = pd.concat(frames, axis=1)
    display(combined)


In [5]:
import pandas as pd

pairs = {
    "full": ("str",     "ud"),
    "old":  ("str-old", "ud-old"),
    "new":  ("str-new", "ud-new"),
}

freq_upos = {}

for name, (split_str, split_ud) in pairs.items():
    df_str = DATA[split_str]
    df_ud  = DATA[split_ud]
    freq_upos[name] = find_most_frequent_errors(df_str, df_ud, feature="upos")

display_side_by_side(freq_upos, n=10)

,full_upos_g_str,full_upos_g_ud,full_upos_p_ud,full_count,old_upos_g_str,old_upos_g_ud,old_upos_p_ud,old_count,new_upos_g_str,new_upos_g_ud,new_upos_p_ud,new_count
0,S,NOUN,PROPN,149,S,NOUN,PROPN,98,S,NOUN,PROPN,27
1,PART,PART,ADV,103,S,PRON,DET,41,S,PROPN,NOUN,23
2,PART,ADV,PART,68,S,PROPN,NOUN,39,V,VERB,AUX,15
3,V,AUX,VERB,56,V,AUX,VERB,32,ADV,ADV,NUM,11
4,S,PROPN,NOUN,55,A,PROPN,ADJ,23,V,AUX,VERB,9
5,S,PRON,DET,54,V,VERB,AUX,18,A,ADJ,VERB,7
6,V,VERB,AUX,47,A,ADJ,VERB,16,S,PRON,DET,7
7,NUM,NUM,ADJ,45,A,ADJ,PROPN,14,ADV,ADV,ADJ,6
8,A,PROPN,ADJ,34,NUM,ADJ,NUM,13,S,ADV,PRON,5
9,S,DET,PRON,30,S,DET,PRON,13,V,VERB,ADJ,5


In [6]:
import dataframe_image as dfi
upos_error_matrix = build_upos_error_matrix(freq_upos, min_total_count=1)
# dfi.export(
#     upos_error_matrix.head(20),
#     "error_matrix.png",
#     table_conversion="matplotlib",
#     dpi=300,
#     fontsize=12, 
# )

### 3. Ошибка PRON <-> DET

In [7]:
errors_S_PRON_DET_full = get_error_sentences(
    DATA["str"], DATA["ud"],
    upos_g_str="S", upos_g_ud="PRON", upos_p_ud="DET")
errors_S_PRON_DET_old = get_error_sentences(
    DATA["str-old"], DATA["ud-old"],
    upos_g_str="S", upos_g_ud="PRON", upos_p_ud="DET")
errors_S_PRON_DET_new = get_error_sentences(
    DATA["str-new"], DATA["ud-new"],
    upos_g_str="S", upos_g_ud="PRON", upos_p_ud="DET")

errors_S_DET_PRON_full = get_error_sentences(
    DATA["str"], DATA["ud"],
    upos_g_str="S", upos_g_ud="DET", upos_p_ud="PRON")
errors_S_DET_PRON_old = get_error_sentences(
    DATA["str-old"], DATA["ud-old"],
    upos_g_str="S", upos_g_ud="DET", upos_p_ud="PRON")
errors_S_DET_PRON_new = get_error_sentences(
    DATA["str-new"], DATA["ud-new"],
    upos_g_str="S", upos_g_ud="DET", upos_p_ud="PRON")

In [8]:
# browse_upos_errors(errors_S_PRON_DET_new)

### 4. Ошибка VERB <-> AUX

In [9]:
errors_V_VERB_AUX_full = get_error_sentences(
    DATA["str"], DATA["ud"],
    upos_g_str="V", upos_g_ud="VERB", upos_p_ud="AUX")
errors_V_VERB_AUX_old = get_error_sentences(
    DATA["str-old"], DATA["ud-old"],
    upos_g_str="V", upos_g_ud="VERB", upos_p_ud="AUX")
errors_V_VERB_AUX_new = get_error_sentences(
    DATA["str-new"], DATA["ud-new"],
    upos_g_str="V", upos_g_ud="VERB", upos_p_ud="AUX")

errors_V_AUX_VERB_full = get_error_sentences(
    DATA["str"], DATA["ud"],
    upos_g_str="V", upos_g_ud="AUX", upos_p_ud="VERB")
errors_V_AUX_VERB_old = get_error_sentences(
    DATA["str-old"], DATA["ud-old"],
    upos_g_str="V", upos_g_ud="AUX", upos_p_ud="VERB")
errors_V_AUX_VERB_new = get_error_sentences(
    DATA["str-new"], DATA["ud-new"],
    upos_g_str="V", upos_g_ud="AUX", upos_p_ud="VERB")

In [24]:
browse_upos_errors(errors_V_AUX_VERB_new)

Всего примеров: 9
Пример #0  (sent_id=148_2019Zanimatelnaya_estetika, id=6)
--------------------------------------------------------------------------------
Предложение:
Я помню, когда мне было семь-восемь лет, я не мог без ужаса смотреть на воронку в ванне, на то, как непостижимо неумолимым образом уходит вода в какое-то неведомое темное пространство под ванной и вообще подо всем, что мы считаем нашим привычным миром.
--------------------------------------------------------------------------------
Токен: 'было'
UPOS:
  SynTagRus gold (upos_g_str): V
  UD gold      (upos_g_ud):  AUX
  UD pred      (upos_p_ud):  VERB
DEPREL:
  SynTagRus: подч-союзн
  UD:        cop
FEATS:
  SynTagRus gold (feats_g_str): Вид=НЕСОВ|Время=ПРОШ|Накл=ИЗЪЯВ|Род=СРЕД|Число=ЕД
  UD gold       (feats_g_ud):  Gender=Neut|Mood=Ind|Number=Sing|Tense=Past|VerbForm=Fin|Voice=Act
  UD pred       (feats_p_ud):  Gender=Neut|Mood=Ind|Number=Sing|Tense=Past|VerbForm=Fin|Voice=Act
Пример #1  (sent_id=170_2020_RFFIMoskva_20